# Task 2.9 — Cross-validation baseline và shortlist DL

**Inputs:** competition data + outputs **2.6 và 2.8**. Kaggle GPU.
5-fold CV so sánh baseline và tối đa 2 cấu hình mới tốt nhất theo holdout. Mỗi fold fit preprocessor riêng và lưu checkpoint tốt nhất.
OOF là xác suất của từng dòng khi dòng đó nằm ở validation fold; không tạo dự đoán test trong notebook này.
**Giới hạn:** shortlist đã được chọn trên holdout và mỗi fold dùng early stopping. Đây là CV phục vụ chọn mô hình, không phải nested CV hoặc điểm test độc lập.


In [ ]:
from pathlib import Path
import importlib.util
import sys
import hashlib
import json

DATA_DIR = None  # Auto-detect inputs; set a directory here only to disambiguate.

def resolve_data_dir(data_dir=None, expected_sha256=None,
                     required_files=("train.csv",), input_root="/kaggle/input"):
    """Find attached data without assuming Kaggle's mount layout."""
    from pathlib import Path
    import hashlib

    root = Path(input_root)
    if data_dir is not None:
        candidates = [Path(data_dir)]
        missing = [name for name in required_files if not (candidates[0] / name).is_file()]
        if missing:
            raise FileNotFoundError(
                f"DATA_DIR={data_dir} is missing {missing}. "
                "Attach competition data or set DATA_DIR=None for automatic discovery."
            )
    else:
        candidates = sorted({
            path.parent for path in root.rglob("train.csv")
            if path.is_file() and all((path.parent / name).is_file() for name in required_files)
        })
        if not candidates:
            mounted = sorted(path.name for path in root.iterdir()) if root.is_dir() else []
            raise FileNotFoundError(
                f"No attached directory contains {list(required_files)} under {root}. "
                "Use Add Input to attach competition playground-series-s6e11. "
                f"Mounted entries: {mounted}"
            )

    if expected_sha256 is not None:
        matching = []
        for directory in candidates:
            digest = hashlib.sha256()
            with (directory / "train.csv").open("rb") as stream:
                for block in iter(lambda: stream.read(1024 * 1024), b""):
                    digest.update(block)
            if digest.hexdigest() == expected_sha256:
                matching.append(directory)
        if not matching:
            raise ValueError(
                "Attached train.csv does not match the task 2.6 checksum. "
                f"Check the competition data/version. Candidates: {candidates}"
            )
        candidates = matching
    elif len(candidates) > 1:
        bundles = [directory for directory in candidates
                   if all((directory / name).is_file()
                          for name in ("train.csv", "test.csv", "sample_submission.csv"))]
        if bundles:
            candidates = bundles

    if len(candidates) != 1:
        raise ValueError(
            f"Multiple valid data directories found: {candidates}. "
            "Set DATA_DIR to the exact directory you want to use."
        )
    return candidates[0]

# Nếu gắn nhiều version 2.6, điền chính xác thư mục version cần dùng.
FOUNDATION_DIR = None
if FOUNDATION_DIR is None:
    found = sorted(Path("/kaggle/input").rglob("foundation_manifest.json"))
    if len(found) != 1:
        raise FileNotFoundError(f"Attach exactly one task 2.6 output; found: {found}")
    FOUNDATION_DIR = found[0].parent
else:
    FOUNDATION_DIR = Path(FOUNDATION_DIR)
manifest = json.loads((FOUNDATION_DIR / "foundation_manifest.json").read_text(encoding="utf-8"))
engine_file = FOUNDATION_DIR / "credit_fraud_dl_engine.py"
if hashlib.sha256(engine_file.read_bytes()).hexdigest() != manifest["engine_sha256"]:
    raise ValueError("Foundation engine checksum mismatch.")
spec = importlib.util.spec_from_file_location("credit_fraud_dl_engine", engine_file)
dl = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = dl
spec.loader.exec_module(dl)

import numpy as np
import pandas as pd
DATA_DIR = resolve_data_dir(DATA_DIR, expected_sha256=manifest["train_sha256"])
print("Resolved DATA_DIR:", DATA_DIR)
train_path = DATA_DIR / "train.csv"
dl.verify_source(train_path, manifest)
train = pd.read_csv(train_path)
dl.validate_train(train)
for name, digest in manifest["membership_sha256"].items():
    if dl.file_hash(FOUNDATION_DIR / name) != digest:
        raise ValueError(f"Foundation membership file changed: {name}")
split = pd.read_csv(FOUNDATION_DIR / "holdout_split_seed9.csv")
train_idx, valid_idx = dl.holdout_indices(train, split)
print("Foundation:", manifest["foundation_id"])
print("Runtime:", dl.environment())


## 2.9.1 — Shortlist và kiểm tra folds

Baseline gốc luôn là control. Tự chọn hai config khác baseline có AUC holdout cao nhất; có thể đặt SHORTLIST_IDS trước khi chạy để kiểm chứng một danh sách đã chốt.
Các model dùng cùng fold IDs và training seed 9. Fold file được checksum bởi 2.6.


In [ ]:
OUTPUT_DIR = Path("/kaggle/working/task_2_9_cross_validation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MAX_CANDIDATES = 2
SHORTLIST_IDS = None  # Ví dụ một danh sách 2 config_id đã có trong feature_manifest.
features_result = dl.json_read(dl.find_input("feature_manifest.json"))
if features_result["foundation_id"] != manifest["foundation_id"]:
    raise ValueError("Feature experiments and CV use different foundations.")
baseline_cfg = dl.config()
baseline_id = dl.config_id(baseline_cfg)
available = {r["config_id"]: r for r in features_result["results"]}
ordered = dl.ranked(list(available.values()))
if SHORTLIST_IDS is None:
    shortlist_ids = [r["config_id"] for r in ordered if r["config_id"] != baseline_id][:MAX_CANDIDATES]
else:
    shortlist_ids = list(dict.fromkeys(SHORTLIST_IDS))
    if len(shortlist_ids) > MAX_CANDIDATES or any(k not in available for k in shortlist_ids):
        raise ValueError("Shortlist must reference at most two completed screening configs.")
candidates = {baseline_id: baseline_cfg}
candidates.update({key: available[key]["config"] for key in shortlist_ids})

membership = pd.read_csv(FOUNDATION_DIR / "cv_folds_seed9.csv",
                         dtype={"feature_group": str})
if membership[dl.ID].duplicated().any() or set(membership[dl.ID]) != set(train[dl.ID]):
    raise ValueError("CV IDs differ from train.")
membership = membership.set_index(dl.ID).loc[train[dl.ID]].reset_index()
fold_ids = membership["fold"].to_numpy(dtype=np.int64)
if set(fold_ids) != set(range(5)):
    raise ValueError("Exactly five complete CV folds are required.")
print("Shortlist IDs:", list(candidates))


## 2.9.2 — Train theo fold, lưu OOF và artifact

Mỗi run chỉ thấy nhãn train của fold để tối ưu loss; AUC validation của fold dùng chọn checkpoint/early stopping.
Checkpoint và preprocessor được lưu theo config_id/fold. Model train xong giải phóng GPU trước lượt sau.
Run hoàn tất có thể reuse khi chạy lại cùng output; nếu mở phiên Kaggle mới cần tự gắn/copy artifact phiên trước để reuse, không tự nhận nhầm output cũ.


In [ ]:
all_fold_rows = []
cv_models = {}
for key, cfg in candidates.items():
    oof = np.full(len(train), np.nan, dtype=np.float64)
    fold_records = []
    fold_paths = []
    fold_artifacts = []
    for fold in range(5):
        fit_idx = np.flatnonzero(fold_ids != fold)
        val_idx = np.flatnonzero(fold_ids == fold)
        if set(membership.iloc[fit_idx]["feature_group"]) & set(membership.iloc[val_idx]["feature_group"]):
            raise ValueError("Feature groups cross a CV partition.")
        relative_dir = f"{key}/fold_{fold}"
        result = dl.train_run(
            train, fit_idx, val_idx, cfg, OUTPUT_DIR / relative_dir, manifest["train_sha256"])
        pred = pd.read_csv(OUTPUT_DIR / relative_dir / "validation.csv")
        expected_ids = train.iloc[val_idx][dl.ID].to_numpy()
        if pred[dl.ID].duplicated().any() or set(pred[dl.ID]) != set(expected_ids):
            raise ValueError("Fold predictions have missing or duplicate IDs.")
        pred = pred.set_index(dl.ID).loc[expected_ids]
        if not np.array_equal(pred[dl.TARGET].to_numpy(), train.iloc[val_idx][dl.TARGET].to_numpy()):
            raise ValueError("Fold labels differ from source.")
        oof[val_idx] = dl.check_probability(pred["probability"], len(val_idx))
        fold_records.append({"fold": fold, "roc_auc": result["roc_auc"],
                             "average_precision": result["average_precision"],
                             "best_epoch": result["best_epoch"],
                             "training_seconds": result["training_seconds"]})
        fold_paths.append(relative_dir)
        fold_artifacts.append({
            "run_directory": relative_dir, "run_signature": result["run_signature"],
            "metrics_sha256": dl.file_hash(OUTPUT_DIR / relative_dir / "metrics.json"),
        })
        all_fold_rows.append({"config_id": key, **fold_records[-1]})
        pd.DataFrame(all_fold_rows).to_csv(OUTPUT_DIR / "fold_scores.csv", index=False)
    dl.check_probability(oof, len(train))
    oof_file = f"{key}_oof.csv"
    pd.DataFrame({dl.ID: train[dl.ID], dl.TARGET: train[dl.TARGET],
                  "fold": fold_ids, "probability": oof}).to_csv(OUTPUT_DIR / oof_file, index=False)
    aucs = np.array([r["roc_auc"] for r in fold_records])
    aps = np.array([r["average_precision"] for r in fold_records])
    cv_models[key] = {
        "config": cfg, "folds": fold_records, "fold_paths": fold_paths,
        "fold_artifacts": fold_artifacts,
        "oof_file": oof_file, "oof_sha256": dl.file_hash(OUTPUT_DIR / oof_file),
        "mean_fold_auc": float(aucs.mean()), "std_fold_auc": float(aucs.std(ddof=1)),
        "mean_fold_ap": float(aps.mean()), "pooled_oof": dl.scores(train[dl.TARGET], oof),
    }

baseline_folds = np.array([r["roc_auc"] for r in cv_models[baseline_id]["folds"]])
comparison = []
for key, record in cv_models.items():
    deltas = np.array([r["roc_auc"] for r in record["folds"]]) - baseline_folds
    record["mean_auc_delta_vs_baseline"] = float(deltas.mean())
    record["folds_better_than_baseline"] = int((deltas > 0).sum())
    comparison.append({
        "config_id": key, "family": record["config"]["family"],
        "mean_fold_auc": record["mean_fold_auc"], "std_fold_auc": record["std_fold_auc"],
        "mean_fold_ap": record["mean_fold_ap"],
        "pooled_oof_auc": record["pooled_oof"]["roc_auc"],
        "delta_vs_baseline": record["mean_auc_delta_vs_baseline"],
        "folds_better_than_baseline": record["folds_better_than_baseline"],
    })
cv_manifest = {
    "foundation_id": manifest["foundation_id"], "source_sha256": manifest["train_sha256"],
    "engine_sha256": manifest["engine_sha256"], "fold_count": 5,
    "baseline_id": baseline_id, "models": cv_models,
    "cv_method": manifest["cv_method"],
    "limitations": "Non-nested model-selection CV; checkpoint selection uses fold validation; "
                   "screening used labels in the earlier holdout. Not an unbiased test estimate.",
}
pd.DataFrame(comparison).sort_values("mean_fold_auc", ascending=False).to_csv(
    OUTPUT_DIR / "cv_comparison.csv", index=False)
dl.json_write(OUTPUT_DIR / "cv_manifest.json", cv_manifest)
display(pd.read_csv(OUTPUT_DIR / "cv_comparison.csv"))
